# HW5: Dyna-Q — Integrating Planning, Acting, and Learning

> - Full Name: **[Full Name]**
> - Student ID: **[Student ID]**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/DeepRLCourse/Homework-5-Questions/blob/main/RL_HW5_Dyna.ipynb)
[![Open In kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://raw.githubusercontent.com/DeepRLCourse/Homework-5-Questions/main/RL_HW5_Dyna.ipynb)

## Overview

Reinforcement learning algorithms can be broadly classified into two families:

- **Model-free methods** (e.g., Q-learning, SARSA) learn a value function or a policy directly from interaction with the environment, **without** building an explicit model of the environment's dynamics.
- **Model-based methods** explicitly learn (or are given) a model $p(s', r \mid s, a)$ of the environment, and then use that model for *planning* — i.e., computing improved values or policies via simulated experience.

**Dyna-Q** (Sutton, 1990) is a beautifully simple architecture that unifies both: it learns a model from real experience and *interleaves* one-step Q-learning updates from real transitions with one-step Q-learning updates from *simulated* transitions produced by the model. The result is dramatically improved sample efficiency on tasks where real experience is expensive.

In this assignment you will:

1. Familiarize yourself with the [**Cliff Walking**](https://gymnasium.farama.org/environments/toy_text/cliff_walking/) environment.
2. Implement $\varepsilon$-greedy and greedy policies.
3. Implement **Dyna-Q** for a deterministic environment.
4. Investigate how the number of planning steps affects learning.
5. Improve learning with **reward shaping**.
6. Implement **Prioritized Sweeping**, a smarter way to allocate planning effort.
7. (Bonus) Extend everything to the larger, stochastic **Taxi-v3** environment.

Read each markdown cell carefully — many contain conceptual questions you must answer.

In [1]:
# @title Imports

import random
import numpy as np
import gymnasium as gym
from tqdm.notebook import trange
from heapq import heappush, heappop
from collections import defaultdict

import matplotlib
from matplotlib import pyplot as plt
import matplotlib.patches as patches
from matplotlib.gridspec import GridSpec

import base64
import imageio
import IPython

import seaborn as sns

In [2]:
# @title Visualization Functions

def embed_mp4(filename):
    video = open(filename, 'rb').read()
    b64 = base64.b64encode(video)
    tag = '''
    <video width="640" height="480" controls>
    <source src="data:video/mp4;base64,{0}" type="video/mp4">
    Your browser does not support the video tag.
    </video>'''.format(b64.decode())
    return IPython.display.HTML(tag)


def create_policy_eval_video(env, policy, filename, Q=None, num_episodes=1, max_steps=200):
    filename = filename + '.mp4'
    with imageio.get_writer(filename, fps=env.metadata.get('render_fps', 4)) as video:
        for _ in range(num_episodes):
            state, info = env.reset()
            video.append_data(env.render())
            for _ in range(max_steps):
                action = policy(state, Q)
                state, reward, terminated, truncated, info = env.step(action)
                video.append_data(env.render())
                if terminated or truncated:
                    break
    return embed_mp4(filename)


def plot_rewards(rewards, average_range=None, ax=None, show=False):
    if ax is None:
        fig, ax = plt.subplots()
    xs = range(1, len(rewards) + 1)
    ax.plot(xs, rewards, marker='.', linestyle='--', alpha=0.4, label='Episode Reward')
    ax.plot(xs, np.cumsum(rewards) / xs, label='Cumulative Average')
    if len(rewards) >= 20:
        window = min(50, len(rewards) // 5)
        smooth = np.convolve(rewards, np.ones(window)/window, mode='valid')
        ax.plot(range(window, len(rewards) + 1), smooth, label=f'Rolling Mean (w={window})')
    ax.legend()
    ax.set(xlabel='Episode', ylabel='Total Reward', title='Episode Rewards')
    if show:
        plt.show()


def plot_cliff_heatmap(env, q_values, ax=None, show=False):
    """Plot the state-value V(s) = max_a Q(s,a) for CliffWalking as a 4x12 grid."""
    if ax is None:
        fig, ax = plt.subplots(figsize=(10, 3))
    rows, cols = 4, 12
    v = q_values.max(axis=1).reshape(rows, cols)
    best_a = q_values.argmax(axis=1)
    act_arrows = {0: '↑', 1: '→', 2: '↓', 3: '←'}
    labels = np.array([act_arrows[a] for a in best_a]).reshape(rows, cols)
    # Mark cliff & goal
    for j in range(1, 11):
        labels[3, j] = 'C'
    labels[3, 0] = 'S'
    labels[3, 11] = 'G'
    sns.heatmap(v, cmap='RdYlGn', annot=labels, fmt='s', ax=ax, cbar_kws={'label': 'V(s)'})
    ax.set_title('Greedy Policy & State Values')
    if show:
        plt.show()


def plot_performance(env, q_values, reward_sums):
    fig = plt.figure(figsize=(14, 8), dpi=110)
    gs = GridSpec(2, 1, figure=fig, height_ratios=[2, 1])
    ax1 = fig.add_subplot(gs[0, 0])
    plot_rewards(reward_sums, ax=ax1)
    ax2 = fig.add_subplot(gs[1, 0])
    plot_cliff_heatmap(env, q_values, ax=ax2)
    plt.tight_layout()
    plt.show()

# 1. Explore the Environment (5 points)

## The Cliff Walking Problem

**Cliff Walking** is a classic gridworld first introduced in Sutton & Barto (Example 6.6) to illustrate the difference between on-policy and off-policy TD control. The world is a $4 \times 12$ grid:

- The agent starts at the bottom-left corner **S**.
- The goal **G** is at the bottom-right corner.
- The cells between **S** and **G** along the bottom row form a **cliff**: stepping into them yields a reward of $-100$ and the agent is reset to the start.
- Every other transition yields a reward of $-1$.
- Actions: `0=Up, 1=Right, 2=Down, 3=Left`.
- The environment is **deterministic**.

Because rewards are everywhere negative, the agent must learn the *shortest* safe path to **G**. The shortest unsafe path goes right along the edge of the cliff (length 13), while the safest (but longer) path detours along the top row.

**Question 1.1.** Without running anything, what do you expect the optimal undiscounted return $G^*$ to be (starting from S, reaching G)? Show your reasoning.

`Your Answer:`

**Answer (1.1).** The optimal undiscounted return is **G\* = -13**. The shortest *safe* path goes up one cell out of the start, straight across the row just above the cliff, then down into the goal: that is 13 moves, each costing -1, and it never enters a -100 cliff cell. So the best achievable return is 13 x (-1) = **-13**.

In [ ]:
env = gym.make('CliffWalking-v0', render_mode='rgb_array')

# Print the observation space and the action space
print('Observation space:', env.observation_space)
print('Action space:', env.action_space)
print('Number of states:', env.observation_space.n)
print('Number of actions:', env.action_space.n)


Define a uniform random policy.

In [ ]:
def random_policy(*args):
    # Select a uniformly random action (CliffWalking has 4 actions)
    action = np.random.randint(4)
    return action


Visualize the random policy. (Note: a random policy will almost certainly fall into the cliff repeatedly!)

In [ ]:
create_policy_eval_video(env, random_policy, 'random_policy', num_episodes=2, max_steps=50)

# 2. Policies (5 points)

## Exploration vs. Exploitation

Any control algorithm must balance two competing pressures:

- **Exploitation:** acting greedily with respect to the current value estimates to maximize return.
- **Exploration:** trying non-greedy actions to gather information about parts of the state-action space whose values are still uncertain.

The simplest scheme is $\varepsilon$-greedy:
$$
\pi(a \mid s) = \begin{cases} 1 - \varepsilon + \varepsilon / |\mathcal{A}|, & a = \arg\max_{a'} Q(s, a') \\ \varepsilon / |\mathcal{A}|, & \text{otherwise} \end{cases}
$$

**Question 2.1.** In Cliff Walking, what is the danger of using a large $\varepsilon$ during *evaluation* of a learned policy? What is the danger of using too small an $\varepsilon$ during *learning*?

`Your Answer:`

**Answer (2.1).** A large epsilon during *evaluation* makes the agent take random actions right next to the cliff, so it repeatedly falls in (-100) even though its greedy Q-values are good - evaluation return becomes poor and high-variance. A too-small epsilon during *learning* means insufficient exploration: many state-action pairs are visited rarely or never, so Q-learning is slow to find the optimal path and can get stuck in a suboptimal policy.

In [ ]:
def greedy_policy(state: int, q_values: np.ndarray) -> int:
    # Return argmax_a Q(state, a)
    action = int(np.argmax(q_values[state]))
    return action


In [ ]:
def epsilon_greedy_policy(state: int, q_values: np.ndarray, epsilon: float) -> int:
    # With probability epsilon take a random action; otherwise act greedily.
    if np.random.random() < epsilon:
        action = np.random.randint(q_values.shape[1])
    else:
        action = int(np.argmax(q_values[state]))
    return action


# 3. Dyna-Q (25 points)

## The Dyna Architecture

Dyna-Q maintains three things in parallel:

1. **A value function** $Q(s,a)$, updated by tabular one-step Q-learning.
2. **A model** of the environment. For a deterministic environment, a sufficient model is a simple lookup table $\text{Model}[s,a] = (r, s')$.
3. **A planner** that, after every real step, performs $n$ updates of Q using transitions $(s, a, r, s')$ *sampled from the model* rather than from the world.

The full update rules are:

**Direct RL (from real experience):**
$$
Q(s_t, a_t) \leftarrow Q(s_t, a_t) + \alpha\Big[r_t + \gamma \max_{a'} Q(s_{t+1}, a') - Q(s_t, a_t)\Big]
$$

**Model learning (deterministic):**
$$
\text{Model}[s_t, a_t] \leftarrow (r_t, s_{t+1})
$$

**Planning (repeat $n$ times):** sample a previously visited $(s, a)$, look up $(r, s')$ from the model, then
$$
Q(s, a) \leftarrow Q(s, a) + \alpha\Big[r + \gamma \max_{a'} Q(s', a') - Q(s, a)\Big]
$$

**Question 3.1.** Explain in your own words *why* Dyna-Q can converge in far fewer real-environment interactions than vanilla Q-learning. What is the computational cost?

**Question 3.2.** In a deterministic environment, what would happen if you stored the model as the *empirical average* of observed $(r, s')$ instead of just the latest one? Would it matter here?

`Your Answers:`

**Answer (3.1).** Each real transition is used to update an internal model, and the planner then replays many *simulated* transitions, propagating reward information backward through already-visited states without any new environment interaction. So one real step yields n+1 Q-updates instead of 1, and value information travels across the state space far faster - dramatically fewer real interactions are needed. The cost is computational: n extra Q-updates per real step plus memory to store the model.

**Answer (3.2).** In a deterministic environment a given (s, a) always yields the same (r, s'), so the empirical average equals the latest observation - it makes no difference here. Storing the average only matters in a *stochastic* environment, where the latest sample is a noisy estimate of the true distribution.

## 3.1 Planning (10 points)

Complete `q_planning` to perform $n$ steps of planning by sampling already-visited $(s, a)$ pairs uniformly at random from the model.

In [ ]:
def q_planning(model: dict, q: np.ndarray, alpha: float, gamma: float, n: int) -> np.ndarray:
    """Perform n planning updates from a deterministic model.

    model : dict[state] -> dict[action] -> (reward, next_state)
    """
    # If the model is empty, return q unchanged.
    if len(model) == 0:
        return q

    states = list(model.keys())
    for _ in range(n):
        # 1. Sample a visited state s uniformly.
        s = states[np.random.randint(len(states))]
        # 2. Sample a visited action a uniformly.
        actions = list(model[s].keys())
        a = actions[np.random.randint(len(actions))]
        # 3. Retrieve (r, s') from the model.
        r, s_next = model[s][a]
        # 4. Q-learning update from the simulated transition.
        q[s, a] += alpha * (r + gamma * np.max(q[s_next]) - q[s, a])

    return q


## 3.2 Learning (15 points)

Now put everything together. At each real step you must:

1. Select an action with $\varepsilon$-greedy.
2. Step the real environment.
3. Perform the direct Q-learning update.
4. Update the model.
5. Perform $n$ planning steps.

Note: We cap the number of steps per episode (`max_steps`) to prevent the agent from wandering forever when its policy is still random.

In [ ]:
def dyna_q(n_episodes: int, env: gym.Env, epsilon: float, alpha: float,
           gamma: float, n: int, max_steps: int = 200) -> tuple[np.ndarray, np.ndarray]:
    """Dyna-Q for a deterministic environment."""

    reward_sums = np.zeros(n_episodes)
    q = np.zeros((env.observation_space.n, env.action_space.n))
    model = defaultdict(dict)

    for episode_i in (pbar := trange(n_episodes, leave=False)):
        state, info = env.reset()
        reward_sum, terminal, steps = 0.0, False, 0

        while not terminal and steps < max_steps:
            # 1. Select an action via epsilon-greedy.
            action = epsilon_greedy_policy(state, q, epsilon)

            # 2. Step the environment.
            next_state, reward, terminated, truncated, info = env.step(action)
            terminal = terminated or truncated

            # 3. Direct Q-learning update.
            #    Bootstrap only when the episode did not truly terminate
            #    (time-limit truncation should still bootstrap).
            bootstrap = 0.0 if terminated else np.max(q[next_state])
            q[state, action] += alpha * (reward + gamma * bootstrap - q[state, action])

            # 4. Update the deterministic model.
            model[state][action] = (reward, next_state)

            # 5. Planning step(s).
            q = q_planning(model, q, alpha, gamma, n)

            # 6. Bookkeeping.
            state = next_state
            reward_sum += reward
            steps += 1

        pbar.set_description(f'Episode {episode_i}, R={reward_sum:.0f}')
        reward_sums[episode_i] = reward_sum

    return q, reward_sums


# 4. Experiments (15 points)

Run Dyna-Q with several values of the planning parameter $n$ (e.g. $n \in \{0, 5, 50\}$).
Note that **$n = 0$ recovers pure Q-learning**.

After running, answer the following:

**Question 4.1.** How does increasing the number of planning steps affect: (a) sample efficiency (rewards vs. episodes), and (b) wall-clock time per episode?

**Question 4.2.** Suppose the *first* successful trajectory occurs at episode $N_1$. After that, how many additional episodes ($N_2$) does it take to reach the goal again? Explain how $n$ affects each of $N_1$ and $N_2$, and *why* the effects differ.

**Question 4.3.** What happens to the *learned path* (cliff-edge vs. safer detour) as you vary $\varepsilon$? Why? Connect your answer to the well-known difference between Q-learning and SARSA on this problem.

**Question 4.4.** Cliff Walking is fully deterministic. Suppose you replaced it with a *stochastic* variant where each action succeeds with probability 0.8 and otherwise slips perpendicular. What failure modes would the current **deterministic** Dyna-Q exhibit? Propose how you would modify the model to fix this.

`Your Answers:`

**Answer (4.1).** (a) Larger n improves sample efficiency: value propagates through the model faster, so fewer episodes are needed to reach a good policy. (b) Wall-clock time per episode grows roughly linearly with n, because each real step triggers n planning updates.

**Answer (4.2).** N1 (the first successful trajectory) is governed mostly by exploration (epsilon) and is largely *independent of n* - you still have to physically stumble onto the goal once. After that, n strongly reduces N2: once the model contains a path to the goal, planning back-propagates the goal value so the agent re-reaches it almost immediately. The effects differ because planning can only help *after* the model has seen the reward; before the first success there is nothing useful to plan over.

**Answer (4.3).** Q-learning is off-policy, so it learns the optimal *greedy* policy - the risky path hugging the cliff edge - regardless of epsilon; but with larger epsilon the *behavior* policy falls off the cliff more often, lowering online return. This is the classic Q-learning vs SARSA difference: SARSA is on-policy and accounts for exploratory falls, so it learns a *safer* detour away from the edge, whereas Q-learning's learned path stays on the optimal-but-dangerous edge.

**Answer (4.4).** The deterministic model keeps only the most recent (r, s') per (s, a), so under stochastic dynamics it overwrites with whatever happened last; planning then trusts a single, possibly unrepresentative outcome, giving biased and oscillating Q-values. Fix: make the model stochastic - store counts N(s,a,s') and reward sums to estimate p(s'|s,a) and mean reward, and sample (r, s') from those estimates (or do an expected update) during planning.

In [ ]:
np.random.seed(2025)
random.seed(2025)

params = {
    'epsilon': 0.1,    # epsilon-greedy exploration rate
    'alpha':   0.5,    # learning rate (tabular, deterministic env => can be large)
    'gamma':   0.99,   # discount factor
    'n':       10,     # number of planning steps per real step
}

n_episodes = 500

env = gym.make('CliffWalking-v0')

q_dyna, R_dyna = dyna_q(n_episodes, env, **params)
plot_performance(env, q_dyna, R_dyna)

**Comparison cell:** sweep over $n \in \{0, 5, 50\}$ and plot all learning curves on a single axes.

In [ ]:
# Sweep over n in {0, 5, 50}; n=0 recovers pure Q-learning.
n_episodes = 500
fig, ax = plt.subplots(figsize=(10, 5))

for n in [0, 5, 50]:
    np.random.seed(2025)
    random.seed(2025)
    env_n = gym.make('CliffWalking-v0')
    _, R = dyna_q(n_episodes, env_n, epsilon=0.1, alpha=0.5, gamma=0.99, n=n)

    window = min(50, max(1, len(R) // 5))
    smooth = np.convolve(R, np.ones(window) / window, mode='valid')
    ax.plot(range(window, len(R) + 1), smooth, label=f'n={n}')

ax.set(xlabel='Episode', ylabel=f'Rolling-mean reward (w<=50)',
       title='Dyna-Q on Cliff Walking: effect of planning steps n')
ax.set_ylim(-200, 0)
ax.legend()
ax.grid(alpha=0.3)
plt.show()


# 5. Improving Performance (15 points)

Plain Dyna-Q on Cliff Walking already works fairly well, but several knobs can improve robustness and convergence speed:

1. **Optimistic initialization.** Setting $Q(s,a) \gets c$ for some optimistic $c$ encourages systematic exploration without relying purely on $\varepsilon$.
2. **Decaying $\varepsilon$.** Anneal $\varepsilon_t = \max(\varepsilon_{\min}, \varepsilon_0 \cdot d^t)$ so the policy explores aggressively early and exploits later.
3. **Decaying $\alpha$.** Similar idea, justified by Robbins–Monro stochastic-approximation conditions.
4. **Boltzmann (softmax) exploration** instead of $\varepsilon$-greedy.
5. **Adaptive planning budget.** Increase $n$ only after the agent has accumulated enough model entries.

Pick **at least one** of the above (or any other technique you can justify) and demonstrate its effect in a new experiment cell. **Do not overwrite your previous experiments** — leave them for comparison.

**Question 5.1.** Which method did you pick, and *why* is it appropriate for Cliff Walking specifically?

`Your Answer:`

**Answer (5.1).** I chose **decaying epsilon**. Cliff Walking needs aggressive early exploration to discover any path across the cliff, but once a good path is known, a high fixed epsilon keeps shoving the agent off the edge (-100 each time), which depresses online return. Annealing epsilon from 0.5 down to 0.01 keeps early exploration high to find the goal, then lets the agent exploit the safe/optimal path, improving both online return and stability - exactly the failure mode Cliff Walking is designed to expose.

In [ ]:
# Improvement: decaying epsilon (explore early, exploit late).
def dyna_q_decaying_eps(n_episodes, env, epsilon0, epsilon_min, decay,
                        alpha, gamma, n, max_steps=200):
    reward_sums = np.zeros(n_episodes)
    q = np.zeros((env.observation_space.n, env.action_space.n))
    model = defaultdict(dict)
    epsilon = epsilon0

    for episode_i in (pbar := trange(n_episodes, leave=False)):
        state, info = env.reset()
        reward_sum, terminal, steps = 0.0, False, 0

        while not terminal and steps < max_steps:
            action = epsilon_greedy_policy(state, q, epsilon)
            next_state, reward, terminated, truncated, info = env.step(action)
            terminal = terminated or truncated

            q[state, action] += alpha * (reward + gamma * np.max(q[next_state]) - q[state, action])
            model[state][action] = (reward, next_state)
            q = q_planning(model, q, alpha, gamma, n)

            state = next_state
            reward_sum += reward
            steps += 1

        epsilon = max(epsilon_min, epsilon * decay)
        pbar.set_description(f'Ep {episode_i}, eps={epsilon:.3f}, R={reward_sum:.0f}')
        reward_sums[episode_i] = reward_sum

    return q, reward_sums


n_episodes = 500

# Baseline: fixed epsilon Dyna-Q (n=10).
np.random.seed(2025); random.seed(2025)
env_base = gym.make('CliffWalking-v0')
_, R_fixed = dyna_q(n_episodes, env_base, epsilon=0.1, alpha=0.5, gamma=0.99, n=10)

# Improvement: decaying epsilon Dyna-Q (n=10).
np.random.seed(2025); random.seed(2025)
env_dec = gym.make('CliffWalking-v0')
_, R_decay = dyna_q_decaying_eps(n_episodes, env_dec, epsilon0=0.5, epsilon_min=0.01,
                                 decay=0.99, alpha=0.5, gamma=0.99, n=10)

fig, ax = plt.subplots(figsize=(10, 5))
for R, lbl in [(R_fixed, 'fixed eps=0.1'), (R_decay, 'decaying eps 0.5->0.01')]:
    window = min(50, max(1, len(R) // 5))
    smooth = np.convolve(R, np.ones(window) / window, mode='valid')
    ax.plot(range(window, len(R) + 1), smooth, label=lbl)
ax.set(xlabel='Episode', ylabel='Rolling-mean reward',
       title='Improvement: decaying epsilon vs fixed epsilon')
ax.set_ylim(-200, 0)
ax.legend(); ax.grid(alpha=0.3)
plt.show()


# 6. Reward Shaping (10 points)

**Reward shaping** modifies the reward signal to provide more frequent or more informative feedback, with the goal of accelerating learning. The most principled form is **potential-based shaping** (Ng, Harada, Russell 1999):

$$
F(s, a, s') = \gamma \Phi(s') - \Phi(s)
$$

where $\Phi : \mathcal{S} \to \mathbb{R}$ is any potential function. A foundational theorem states:

> *Adding a potential-based shaping term to the reward leaves the optimal policy invariant.*

Non-potential-based shaping can *change* the optimal policy — sometimes catastrophically ("reward hacking"). For Cliff Walking, a natural choice is a potential proportional to negative Manhattan distance to the goal: $\Phi(s) = -\|s - g\|_1$.

**Question 6.1.** Why does potential-based shaping preserve the optimal policy? Sketch the argument.

**Question 6.2.** Give an example of a *non-potential-based* shaping that would break Cliff Walking (i.e., cause the agent to learn a suboptimal policy).

`Your Answers:`

**Answer (6.1).** Potential-based shaping F = gamma\*Phi(s') - Phi(s) telescopes along any trajectory: summing the discounted shaping over an episode collapses to gamma^T\*Phi(s_T) - Phi(s_0), which depends only on the start and terminal states, not on the actions chosen in between. Equivalently, it shifts every Q-value by exactly -Phi(s), so the *argmax* over actions - and hence the greedy/optimal policy - is unchanged.

**Answer (6.2).** A non-potential shaping that breaks it: add +2 every time the agent moves *right*, independent of any potential. The agent can now farm this bonus by hugging the cliff edge (or by dithering to collect rightward steps) instead of minimizing path cost, so the learned optimal policy changes - classic reward hacking.

In [ ]:
from gymnasium.envs.toy_text import CliffWalkingEnv

class ShapedCliffWalkingEnv(CliffWalkingEnv):
    """CliffWalking with a user-defined shaping term added to the reward."""

    GOAL_STATE = 47  # index of the goal cell in the 4x12 grid
    NROWS, NCOLS = 4, 12

    def __init__(self, gamma: float = 0.99, **kwargs):
        super().__init__(**kwargs)
        self.gamma = gamma
        self._prev_state = None

    def reset(self, **kwargs):
        obs, info = super().reset(**kwargs)
        self._prev_state = obs
        return obs, info

    def potential(self, state: int) -> float:
        # Phi(s) = negative Manhattan distance to the goal.
        row, col = divmod(int(state), self.NCOLS)
        grow, gcol = divmod(self.GOAL_STATE, self.NCOLS)
        return -(abs(row - grow) + abs(col - gcol))

    def step(self, action):
        obs, reward, terminated, truncated, info = super().step(action)
        # Potential-based shaping: F = gamma * Phi(s') - Phi(s)
        shaping = self.gamma * self.potential(obs) - self.potential(self._prev_state)
        shaped_reward = reward + shaping
        self._prev_state = obs
        return obs, shaped_reward, terminated, truncated, info


In [ ]:
np.random.seed(2025)
random.seed(2025)

params = {
    'epsilon': 0.1,
    'alpha':   0.5,
    'gamma':   0.99,
    'n':       10,
}
n_episodes = 500

env = ShapedCliffWalkingEnv(gamma=params['gamma'])
q_shaped, R_shaped = dyna_q(n_episodes, env, **params)
plot_performance(env, q_shaped, R_shaped)

**Question 6.3.** Compare the learning curves with and without shaping. Did shaping help in the *early* episodes? In the *late* episodes? Why?

`Your Answer:`

**Answer (6.3).** Shaping provides a denser gradient toward the goal, so the *early* episodes improve noticeably faster - less aimless wandering before the first success. In the *late* episodes the curves converge to the same optimum, because potential-based shaping leaves the optimal policy invariant; the benefit is concentrated in early-learning speed, not final performance.

# 7. Prioritized Sweeping (20 points)

Uniform sampling of $(s, a)$ pairs in the planning loop is wasteful. Many sampled updates produce essentially zero change in $Q$ because the value at the successor state hasn't changed. **Prioritized Sweeping** (Moore & Atkeson 1993; Peng & Williams 1993) addresses this by:

1. Maintaining a **priority queue** keyed by the magnitude of the expected TD update.
2. On each real step, computing the TD error $\delta$ for the current $(s, a)$. If $|\delta| > \theta$ (a threshold), push $(s, a)$ onto the queue with priority $|\delta|$.
3. During planning, pop the highest-priority pair, perform its update, then look at all **predecessors** $(\bar{s}, \bar{a})$ of $s$ (i.e., pairs known by the model to lead to $s$). Compute *their* TD errors; if they exceed $\theta$, push them too.

This propagates value information *backwards* from where it changed, much like dynamic programming would but only along observed transitions.

**Question 7.1.** Why is the magnitude of the TD error a sensible priority? What property does it estimate?

**Question 7.2.** Python's `heapq` is a *min-heap*. How do you use it as a *max-heap* for priorities?

**Question 7.3.** What is the role of $\theta$? What happens if $\theta = 0$? If $\theta$ is very large?

`Your Answers:`

**Answer (7.1).** The magnitude of the TD error estimates how much Q(s,a) would move if updated - i.e. the size of the pending Bellman backup and the local inconsistency between Q(s,a) and its bootstrapped target. A large |delta| means an update here will change the value a lot and is worth doing; near-zero means the update is wasted effort. Prioritizing by |delta| spends planning where it matters most.

**Answer (7.2).** Python's heapq is a min-heap, so to get a max-heap on priority you push the *negative* priority: store (-|delta|, (s, a)). The most negative key (largest |delta|) is then popped first.

**Answer (7.3).** theta is a significance threshold: only (s, a) with |delta| > theta are enqueued. If theta = 0, essentially every nonzero TD error is queued (maximal sweeping, most compute). If theta is very large, almost nothing is enqueued and planning is effectively switched off, reducing the method to plain one-step Q-learning.

## 7.1 Priority Planning (10 points)

Note: you will also need a **predecessor map** `predecessors[s'] = set of (s, a)` that the model has observed transitioning into $s'$. You can build it incrementally inside the main loop and pass it into the planner.

In [ ]:
def q_planning_priority(model: dict, predecessors: dict, q: np.ndarray,
                        priorities: list, alpha: float, gamma: float,
                        n: int, theta: float) -> np.ndarray:
    """Perform up to n planning updates using prioritized sweeping."""

    count = 0
    while priorities and count < n:
        # 1. Pop the highest-priority (s, a). (heapq is a min-heap; we stored -|delta|.)
        _, (s, a) = heappop(priorities)

        # 2. Look up (r, s') and perform the Q-learning update.
        r, s_next = model[s][a]
        q[s, a] += alpha * (r + gamma * np.max(q[s_next]) - q[s, a])

        # 3. For each predecessor of s, compute its TD error and enqueue if large.
        for (s_bar, a_bar) in predecessors[s]:
            r_bar, _ = model[s_bar][a_bar]
            delta_bar = r_bar + gamma * np.max(q[s]) - q[s_bar, a_bar]
            if abs(delta_bar) > theta:
                heappush(priorities, (-abs(delta_bar), (s_bar, a_bar)))

        count += 1

    return q


## 7.2 Learning with Prioritized Sweeping (10 points)

In [ ]:
def dyna_q_priority(n_episodes: int, env: gym.Env, epsilon: float, alpha: float,
                   gamma: float, n: int, theta: float,
                   max_steps: int = 200) -> tuple[np.ndarray, np.ndarray]:
    """Dyna-Q with prioritized sweeping (deterministic model)."""

    reward_sums = np.zeros(n_episodes)
    q = np.zeros((env.observation_space.n, env.action_space.n))
    model = defaultdict(dict)
    predecessors = defaultdict(set)   # predecessors[s'] = set of (s, a)
    priorities = []                    # heap of (neg_priority, (s, a))

    for episode_i in (pbar := trange(n_episodes, leave=False)):
        state, info = env.reset()
        reward_sum, terminal, steps = 0.0, False, 0

        while not terminal and steps < max_steps:
            # 1. Select action via epsilon-greedy.
            action = epsilon_greedy_policy(state, q, epsilon)

            # 2. Step env.
            next_state, reward, terminated, truncated, info = env.step(action)
            terminal = terminated or truncated

            # 3. TD error of the real transition (mask bootstrap on true termination).
            bootstrap = 0.0 if terminated else np.max(q[next_state])
            delta = reward + gamma * bootstrap - q[state, action]

            # 4. Direct Q-learning update.
            q[state, action] += alpha * delta

            # 5. Update model and predecessor map.
            model[state][action] = (reward, next_state)
            predecessors[next_state].add((state, action))

            # 6. Push onto the priority queue if the TD error is significant.
            if abs(delta) > theta:
                heappush(priorities, (-abs(delta), (state, action)))

            # 7. Prioritized planning sweep.
            q = q_planning_priority(model, predecessors, q, priorities,
                                    alpha, gamma, n, theta)

            # 8. Bookkeeping.
            state = next_state
            reward_sum += reward
            steps += 1

        pbar.set_description(f'Episode {episode_i}, R={reward_sum:.0f}')
        reward_sums[episode_i] = reward_sum

    return q, reward_sums


# 8. Final Experiments (5 points)

Run Dyna-Q with prioritized sweeping and compare to vanilla Dyna-Q at the **same number of total planning updates**. Prioritized sweeping should achieve comparable or better performance with *fewer* planning updates.

In [ ]:
np.random.seed(2025)
random.seed(2025)

params = {
    'epsilon': 0.1,
    'alpha':   0.5,
    'gamma':   0.99,
    'n':       10,
    'theta':   1e-3,   # priority threshold
}
n_episodes = 500

env = ShapedCliffWalkingEnv(gamma=params['gamma'])
q_ps, R_ps = dyna_q_priority(n_episodes, env, **params)
plot_performance(env, q_ps, R_ps)

Visualize the learned greedy policy.

In [ ]:
env_vis = ShapedCliffWalkingEnv(gamma=params['gamma'], render_mode='rgb_array')
create_policy_eval_video(env_vis, greedy_policy, 'DynaQ_PS', Q=q_ps, max_steps=50)

**Question 8.1.** Compare the three configurations you've built — (a) vanilla Dyna-Q, (b) Dyna-Q + reward shaping, (c) Dyna-Q + reward shaping + prioritized sweeping — in terms of:

- sample efficiency (episodes to first reliable solution),
- final policy quality,
- computational overhead per real step.

Which is the most cost-effective in your view, and why?

`Your Answer:`

**Answer (8.1).** *(a) Vanilla Dyna-Q* reliably solves Cliff Walking but spreads its n planning updates uniformly, wasting many on states whose values have not changed. *(b) Dyna-Q + reward shaping* speeds up early learning via a denser signal at essentially zero extra cost per step, and converges to the same optimal policy. *(c) Dyna-Q + shaping + prioritized sweeping* reaches a good policy with the *fewest* planning updates because effort is focused where the TD error is largest, at a small overhead for maintaining the priority queue and predecessor map.

In terms of *sample efficiency* PS is best (fewest episodes to a reliable solution at equal planning budget); *final policy quality* is the same optimal -13 path for all three (shaping is potential-based, so it does not change the optimum); *per-step overhead* is lowest for shaping and highest for PS (heap operations). Overall the most cost-effective choice is **prioritized sweeping combined with reward shaping**: shaping accelerates the very first success, and PS then propagates that information through the model with minimal wasted updates.

# 9. Bonus (10 points)

If you'd like to push further, attempt **one** of the following:

**Option A — Stochastic Dyna-Q.** Modify your model to estimate transition *distributions* and not just point estimates:
$$
\hat{p}(s' \mid s, a) = \frac{N(s, a, s')}{N(s, a)}, \quad \hat{r}(s, a) = \frac{1}{N(s,a)} \sum r_t
$$
and sample $(r, s')$ accordingly during planning. Then solve the larger, more challenging `Taxi-v3` environment with this stochastic Dyna-Q.

**Option B — Dyna-Q+ (exploration bonus).** Implement the Dyna-Q+ variant (Sutton & Barto §8.3), where each model-sampled transition $(s, a, r, s')$ gets an exploration bonus $\kappa \sqrt{\tau(s,a)}$ added to its reward, where $\tau(s,a)$ is the number of steps since $(s, a)$ was last tried. Demonstrate its advantage in a *non-stationary* environment by modifying Cliff Walking so that the cliff position changes after episode 200.

Document your design choices clearly and discuss results.

In [ ]:
# @title 🚶‍♂️ Happy Planning 🧠
